# 课程 04 - 工具使用设计模式

在本课中，您将学习使用 Microsoft Agent Framework (Python) 的 AI 代理的<strong>工具使用</strong>设计模式。我们涵盖：

- 使用 `@tool` 装饰器和类型化参数定义函数工具
- 提供工具模式，让模型了解每个工具的功能
- 使用 `approval_mode` 控制工具执行
- 通过 Pydantic 模型和 `response_format` 返回<strong>结构化输出</strong>

方案是一个<strong>旅游预订代理</strong>，可以查询目的地，检查可用性，并检索航班信息。


## 设置


In [ ]:
%pip install agent-framework python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.openai import OpenAIChatCompletionClient

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("LLM_BASE_URL")
deployment_name = os.getenv("LLM_MODEL")

missing = [k for k, v in {
    "LLM_BASE_URL": endpoint,
    "LLM_MODEL": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
import os
# Create the the chat model provider client
client = OpenAIChatCompletionClient(
    model=os.environ["LLM_MODEL"],
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
)

## 使用 @tool 装饰器定义工具

`@tool` 装饰器将普通的 Python 函数转换为代理可以调用的工具。
关键点：

- <strong>文档字符串</strong> 成为模型看到的工具描述。
- <strong>类型注解</strong>（包括带描述的 `Annotated`）定义工具的模式。
- `approval_mode` 控制是否必须在执行前让用户批准每次调用。


In [ ]:
@tool(approval_mode='never_require')
def get_destinations() -> list[str]:
    """获取可用的度假目的地。"""
    return ['Barcelona', 'Paris', 'Berlin', 'Tokyo', 'Sydney', 'New York City']

@tool(approval_mode='never_require')
def check_availability(destination: Annotated[str, 'The destination to check']) -> str:
    """检查某个目的地的预订可用性。"""
    availability = {'Barcelona': 'Available - 3 spots left', 'Paris': 'Available', 'Berlin': 'Sold out', 'Tokyo': 'Available - 1 spot left', 'Sydney': 'Available', 'New York City': 'Available'}
    return availability.get(destination, 'Unknown destination')

@tool(approval_mode='never_require')
def get_flight_info(origin: Annotated[str, 'Origin airport code'], destination: Annotated[str, 'Destination airport code']) -> str:
    """获取两个城市之间的航班信息。"""
    flights = {'LHR-BCN': 'BA 2042, Departs 08:30, Arrives 11:45, $350', 'LHR-CDG': 'AF 1081, Departs 09:15, Arrives 11:30, $280', 'LHR-NRT': 'JL 044, Departs 11:00, Arrives 07:00+1, $890'}
    return flights.get(f'{origin}-{destination}', f'No direct flights from {origin} to {destination}')

## 创建一个拥有多种工具的代理

将所有三种工具传递给客户端，这样模型就可以调用它们中的任意一个来回答用户的问题。


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]
agent = client.as_agent(name='TravelToolAgent', instructions='你是一名旅行代理。使用可用工具回答有关目的地、可订状态和航班的问题。', tools=travel_tools)
response = await agent.run('你们有哪些目的地？哪些仍然可用？')
print(response)

## 使用工具进行结构化输出

通过将 `response_format` 设置为 Pydantic 模型，代理被强制返回一个类型良好的 JSON 对象，而不是自由格式的文本。当下游代码需要以编程方式消费结果时，这很有用。


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int

class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]
structured_agent = client.as_agent(name='StructuredTravelAgent', instructions='你是一个旅行代理。使用可用的工具来查找目的地、检查可用性并获取航班信息。返回结构化的结果。', tools=[get_destinations, check_availability, get_flight_info])
response = await structured_agent.run('我想从伦敦希思罗机场飞往欧洲某个温暖的地方。查一下有哪些可选项。')
if response:
    print(response)

## 工具批准模式

`@tool` 上的 `approval_mode` 参数控制工具调用在执行前是否需要人工批准：

| 模式 | 行为 |
|---|---|
| `"never_require"` | 工具自动运行 — 不需要用户确认。 |
| `"always_require"` | 每次调用都必须得到用户批准后才能执行。 |

对于有副作用的工具（例如预订航班、扣费信用卡），使用 `"always_require"`，以确保有人介入。 


In [ ]:
@tool(approval_mode='always_require')
def book_flight(origin: Annotated[str, 'Origin airport code'], destination: Annotated[str, 'Destination airport code'], passenger_name: Annotated[str, 'Full name of the passenger']) -> str:
    """为乘客预订航班。执行前需要审批。"""
    return f'Flight booked from {origin} to {destination} for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}'
print('工具名称：', book_flight.name)
print('审批模式：', book_flight.approval_mode)

## 总结

在本课中，您学习了如何：

1. 使用带有类型参数和文档字符串的 `@tool` 装饰器<strong>定义工具</strong>，这些文档字符串用作工具模式。
2. <strong>组合多个工具</strong>，以便代理能够按顺序调用它们来回答复杂查询。
3. 通过传递 Pydantic 模型作为 `response_format`，<strong>返回结构化输出</strong>。
4. 使用 `approval_mode` <strong>控制工具审批</strong>，以便在人类监督下执行敏感操作。

这些模式构成了构建可靠、生产就绪代理的基础，这些代理能够安全地与外部系统交互。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
